# Fleet Maintenance Planning — Three Model Versions

This notebook builds three permitted classical-ML pipelines for the Fleet Maintenance Planning competition.

**Competition objective:** minimise total maintenance cost, not classification error.

- `flag = 1`: inspect the vehicle.
- `flag = 0`: do not inspect.
- `budget`: spare-parts value to pre-order; ignored when `flag = 0`.

The notebook creates three Kaggle-ready files:
- `submission_v1_baseline.csv`
- `submission_v2_lightgbm.csv`
- `submission_v3_catboost.csv`

It also compares validation maintenance costs. Keep the validation results and model changes documented for the project report.

## 1. Imports and reproducibility

The allowed model families in the project statement include Logistic Regression, tree ensembles, and gradient boosting (including LightGBM and CatBoost). This notebook uses those classical models only.

In [ ]:
import os, zipfile, warnings, random
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import HistGradientBoostingRegressor
from sklearn.metrics import average_precision_score, roc_auc_score, precision_recall_curve

from lightgbm import LGBMClassifier, LGBMRegressor, early_stopping, log_evaluation
from catboost import CatBoostClassifier, CatBoostRegressor

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
warnings.filterwarnings("ignore")

## 2. Load the competition data

Put `train.csv`, `test.csv`, and `sample_submission.csv` in the notebook working directory. Alternatively, put the uploaded ZIP file there; the cell below extracts it.

In [ ]:
ZIP_PATH = "mlprojectfleet2.zip"

if os.path.exists(ZIP_PATH) and not os.path.exists("train.csv"):
    with zipfile.ZipFile(ZIP_PATH, "r") as z:
        z.extractall(".")

train = pd.read_csv("train.csv")
test = pd.read_csv("test.csv")
sample = pd.read_csv("sample_submission.csv")

sensor_cols = [c for c in train.columns if c.startswith("s")]
# s047 is entirely missing in the supplied test set, so it cannot provide
# useful test-time information. Drop it consistently from train and test.
feature_cols = [c for c in sensor_cols if c in test.columns]
feature_cols = [c for c in feature_cols if not test[c].isna().all()]

X = train[feature_cols].copy()
X_test = test[feature_cols].copy()
y = train["failure"].astype(int)
repair_cost = train["repair_cost"].fillna(0).astype(float)

print("Train:", train.shape, " Test:", test.shape)
print("Sensor features used:", len(feature_cols))
print("Failure count:", int(y.sum()), "/", len(y))
print("Failure rate:", round(y.mean() * 100, 3), "%")
print("Positive repair-cost rows:", int((train.failure == 1).sum()))
print("Test IDs unique:", test.id.is_unique)

## 3. Initial data inspection

The training file has 56,000 rows and 170 sensor columns. Failure is rare, so accuracy alone is misleading. `followup_days` is not included in the feature matrix because it is absent from test data.

Check missingness and the repair-cost distribution before modelling.

In [ ]:
print(train[["failure", "followup_days", "repair_cost"]].describe(include="all"))
print("\nFailure counts:\n", y.value_counts())
print("\nLargest sensor missingness rates:\n",
      train[feature_cols].isna().mean().sort_values(ascending=False).head(15))

plt.figure(figsize=(6, 4))
y.value_counts().sort_index().plot(kind="bar")
plt.xticks([0, 1], ["No failure", "Failure"], rotation=0)
plt.ylabel("Vehicles")
plt.title("Failure class distribution")
plt.tight_layout()
plt.show()

plt.figure(figsize=(7, 4))
train.loc[train.failure == 1, "repair_cost"].clip(upper=3000).hist(bins=40)
plt.xlabel("Repair cost (clipped at 3000 for display)")
plt.ylabel("Failed vehicles")
plt.title("Repair-cost distribution")
plt.tight_layout()
plt.show()

## 4. Validation split and the competition cost function

We use a stratified holdout split so the rare failure class appears in both sets. This is a useful first validation estimate, but it cannot fully reproduce the shift to a different depot.

The cost function below exactly follows the project statement:

- no inspection: actual repair cost;
- inspection: 10 + 0.20 × under-budget amount + 0.02 × over-budget amount.

For healthy vehicles, actual repair cost is zero.

In [ ]:
X_train, X_valid, y_train, y_valid = train_test_split(
    X, y, test_size=0.25, random_state=SEED, stratify=y
)
c_valid = train.loc[y_valid.index, "repair_cost"].fillna(0).to_numpy(dtype=float)

def competition_cost(y_true_cost, flags, budgets):
    y_true_cost = np.asarray(y_true_cost, dtype=float)
    flags = np.asarray(flags, dtype=int)
    budgets = np.asarray(budgets, dtype=float)
    per_vehicle = np.where(
        flags == 1,
        10 + 0.20 * np.maximum(y_true_cost - budgets, 0)
          + 0.02 * np.maximum(budgets - y_true_cost, 0),
        y_true_cost
    )
    return per_vehicle.mean(), per_vehicle.sum()

def choose_policy(expected_repair_cost, budgets_for_flagged, actual_cost, label="model"):
    # Search decision thresholds using validation only.
    # The final chosen threshold is then applied unchanged to test predictions.
    thresholds = np.unique(np.quantile(expected_repair_cost, np.linspace(0.70, 0.999, 100)))
    rows = []
    for threshold in thresholds:
        flags = (expected_repair_cost > threshold).astype(int)
        budgets = np.where(flags == 1, budgets_for_flagged, 0.0)
        mean_cost, total_cost = competition_cost(actual_cost, flags, budgets)
        rows.append((threshold, int(flags.sum()), mean_cost, total_cost))
    results = pd.DataFrame(rows, columns=["threshold", "inspections", "mean_cost", "total_cost"])
    best = results.loc[results.mean_cost.idxmin()]
    print(f"{label}: best threshold={best.threshold:.4f}, "
          f"inspections={int(best.inspections)}, mean cost={best.mean_cost:.4f}")
    return float(best.threshold), results

# Simple budget baseline: 90th percentile of observed repair costs among failed
# training vehicles. Later versions can improve this with conditional quantiles.
global_q90 = float(train.loc[train.failure == 1, "repair_cost"].quantile(0.90))
print("Training failure rate:", y_train.mean())
print("Global 90th-percentile repair budget:", round(global_q90, 2))

## 5. Model Version 1 — Baseline

- Failure model: scaled Logistic Regression with class weighting.
- Repair-cost model: HistGradientBoostingRegressor trained on log repair costs for failed vehicles.
- The expected repair cost is estimated as predicted failure probability × predicted conditional repair cost.
- The inspection threshold is selected by validation maintenance cost, not accuracy.

In [ ]:
v1_failure = Pipeline([
    ("imputer", SimpleImputer(strategy="median", add_indicator=True)),
    ("scaler", StandardScaler()),
    ("model", LogisticRegression(
        max_iter=3000, class_weight="balanced", C=0.5, random_state=SEED
    ))
])
v1_failure.fit(X_train, y_train)
p1 = v1_failure.predict_proba(X_valid)[:, 1]

pos_train_mask = (y_train == 1)
X_train_pos = X_train.loc[pos_train_mask]
log_cost_train = np.log1p(train.loc[X_train_pos.index, "repair_cost"].astype(float))

v1_cost = Pipeline([
    ("imputer", SimpleImputer(strategy="median", add_indicator=True)),
    ("model", HistGradientBoostingRegressor(
        max_iter=250, learning_rate=0.05, max_leaf_nodes=15,
        l2_regularization=5.0, random_state=SEED
    ))
])
v1_cost.fit(X_train_pos, log_cost_train)
r1 = np.maximum(np.expm1(v1_cost.predict(X_valid)), 0)
expected1 = p1 * r1

threshold1, results1 = choose_policy(
    expected1, global_q90, c_valid, "Version 1"
)
flags1 = (expected1 > threshold1).astype(int)
budget1 = np.where(flags1 == 1, global_q90, 0.0)
v1_mean_cost, v1_total_cost = competition_cost(c_valid, flags1, budget1)

print("V1 average precision:", round(average_precision_score(y_valid, p1), 5))
print("V1 ROC-AUC:", round(roc_auc_score(y_valid, p1), 5))
print("V1 validation mean cost:", round(v1_mean_cost, 5))

## 6. Model Version 2 — LightGBM

LightGBM can model nonlinear sensor interactions and handles missing values natively. We use a positive-class weight to help the model learn from the rare failures.

The repair-cost model is trained only on failed vehicles, using log-transformed repair costs to reduce the influence of the long right tail.

In [ ]:
scale_pos = max(1.0, (y_train == 0).sum() / max(1, (y_train == 1).sum()))

v2_failure = LGBMClassifier(
    n_estimators=1200, learning_rate=0.025, num_leaves=15,
    max_depth=-1, min_child_samples=35, subsample=0.85,
    colsample_bytree=0.85, reg_lambda=5.0,
    scale_pos_weight=scale_pos, random_state=SEED,
    verbosity=-1
)
v2_failure.fit(
    X_train, y_train,
    eval_set=[(X_valid, y_valid)],
    callbacks=[early_stopping(100, verbose=False), log_evaluation(0)]
)
p2 = v2_failure.predict_proba(X_valid)[:, 1]

v2_cost = LGBMRegressor(
    n_estimators=700, learning_rate=0.025, num_leaves=7,
    max_depth=3, min_child_samples=12, subsample=0.9,
    colsample_bytree=0.85, reg_lambda=8.0,
    random_state=SEED, verbosity=-1
)
v2_cost.fit(
    X_train_pos, log_cost_train,
    eval_set=[(X_train_pos, log_cost_train)],
    callbacks=[log_evaluation(0)]
)
r2 = np.maximum(np.expm1(v2_cost.predict(X_valid)), 0)
expected2 = p2 * r2

threshold2, results2 = choose_policy(
    expected2, global_q90, c_valid, "Version 2"
)
flags2 = (expected2 > threshold2).astype(int)
budget2 = np.where(flags2 == 1, global_q90, 0.0)
v2_mean_cost, v2_total_cost = competition_cost(c_valid, flags2, budget2)

print("V2 average precision:", round(average_precision_score(y_valid, p2), 5))
print("V2 ROC-AUC:", round(roc_auc_score(y_valid, p2), 5))
print("V2 validation mean cost:", round(v2_mean_cost, 5))

## 7. Model Version 3 — CatBoost + cost-aware decisions

CatBoost is our advanced candidate. It handles missing sensor values natively and can capture nonlinear relationships.

- Failure model: class-balanced CatBoost classifier.
- Repair-cost model: CatBoost regressor trained on failed vehicles, predicting repair cost directly.
- Expected repair loss: predicted failure probability × predicted conditional repair cost.
- Decision threshold: selected using the validation maintenance-cost function.
- Parts budget: initially use the training-set 90th percentile of observed failure repair costs for inspected vehicles.

This is a strong, reproducible third version. A later extension can replace the fixed 90th-percentile budget with conditional quantile models, but we should first establish a reliable baseline and compare validation costs.

In [ ]:
v3_failure = CatBoostClassifier(
    iterations=1100, depth=5, learning_rate=0.03,
    loss_function="Logloss", eval_metric="PRAUC",
    auto_class_weights="Balanced", l2_leaf_reg=8,
    random_seed=SEED, verbose=False, allow_writing_files=False
)
v3_failure.fit(
    X_train, y_train,
    eval_set=(X_valid, y_valid),
    early_stopping_rounds=120, verbose=False
)
p3 = v3_failure.predict_proba(X_valid)[:, 1]

v3_cost_mean = CatBoostRegressor(
    iterations=900, depth=4, learning_rate=0.025,
    loss_function="RMSE", l2_leaf_reg=10,
    random_seed=SEED, verbose=False, allow_writing_files=False
)
v3_cost_mean.fit(
    X_train_pos, train.loc[X_train_pos.index, "repair_cost"].astype(float),
    verbose=False
)
r3 = np.maximum(v3_cost_mean.predict(X_valid), 0)
expected3 = p3 * r3

threshold3, results3 = choose_policy(
    expected3, global_q90, c_valid, "Version 3"
)
flags3 = (expected3 > threshold3).astype(int)
budget3 = np.where(flags3 == 1, global_q90, 0.0)
v3_mean_cost, v3_total_cost = competition_cost(c_valid, flags3, budget3)

print("V3 average precision:", round(average_precision_score(y_valid, p3), 5))
print("V3 ROC-AUC:", round(roc_auc_score(y_valid, p3), 5))
print("V3 validation mean cost:", round(v3_mean_cost, 5))

## 8. Compare all three versions

The key comparison is validation maintenance cost (lower is better). Average precision is a useful secondary diagnostic for the rare failure class, but it is not the competition objective.

In [ ]:
comparison = pd.DataFrame([
    ["V1 Logistic + HistGradientBoosting", average_precision_score(y_valid, p1), roc_auc_score(y_valid, p1), v1_mean_cost, int(flags1.sum())],
    ["V2 LightGBM", average_precision_score(y_valid, p2), roc_auc_score(y_valid, p2), v2_mean_cost, int(flags2.sum())],
    ["V3 CatBoost", average_precision_score(y_valid, p3), roc_auc_score(y_valid, p3), v3_mean_cost, int(flags3.sum())],
], columns=["Version", "Average Precision", "ROC-AUC", "Validation Mean Cost", "Inspections"])
comparison.sort_values("Validation Mean Cost")

## 9. Refit the selected versions on all training data and create submissions

For each version, refit on the complete training set, predict the test set, and apply its validation-selected threshold. Budgets are non-negative and set to zero for unflagged vehicles.

**Important:** the holdout threshold is an estimate. Compare the three submissions on Kaggle and keep a dated record of each submission. Do not use test labels (they are not provided).

In [ ]:
# Full-data feature matrices
X_full = X
X_test_full = X_test
y_full = y
positive_full = (y_full == 1)
X_positive_full = X_full.loc[positive_full]
log_cost_full = np.log1p(train.loc[X_positive_full.index, "repair_cost"].astype(float))

# V1 full fit
v1_failure_full = Pipeline([
    ("imputer", SimpleImputer(strategy="median", add_indicator=True)),
    ("scaler", StandardScaler()),
    ("model", LogisticRegression(max_iter=3000, class_weight="balanced", C=0.5, random_state=SEED))
])
v1_failure_full.fit(X_full, y_full)
v1_cost_full = Pipeline([
    ("imputer", SimpleImputer(strategy="median", add_indicator=True)),
    ("model", HistGradientBoostingRegressor(max_iter=250, learning_rate=0.05, max_leaf_nodes=15,
                                             l2_regularization=5.0, random_state=SEED))
])
v1_cost_full.fit(X_positive_full, log_cost_full)
p1_test = v1_failure_full.predict_proba(X_test_full)[:, 1]
r1_test = np.maximum(np.expm1(v1_cost_full.predict(X_test_full)), 0)
e1_test = p1_test * r1_test

# V2 full fit
v2_failure_full = LGBMClassifier(
    n_estimators=max(200, getattr(v2_failure, "best_iteration_", 0) or 700),
    learning_rate=0.025, num_leaves=15, min_child_samples=35,
    subsample=0.85, colsample_bytree=0.85, reg_lambda=5.0,
    scale_pos_weight=max(1.0, (y_full == 0).sum() / max(1, (y_full == 1).sum())),
    random_state=SEED, verbosity=-1
)
v2_failure_full.fit(X_full, y_full)
v2_cost_full = LGBMRegressor(
    n_estimators=700, learning_rate=0.025, num_leaves=7, max_depth=3,
    min_child_samples=12, subsample=0.9, colsample_bytree=0.85,
    reg_lambda=8.0, random_state=SEED, verbosity=-1
)
v2_cost_full.fit(X_positive_full, log_cost_full)
p2_test = v2_failure_full.predict_proba(X_test_full)[:, 1]
r2_test = np.maximum(np.expm1(v2_cost_full.predict(X_test_full)), 0)
e2_test = p2_test * r2_test

# V3 full fit
v3_failure_full = CatBoostClassifier(
    iterations=max(200, getattr(v3_failure, "best_iteration_", 0) or 700),
    depth=5, learning_rate=0.03, loss_function="Logloss",
    auto_class_weights="Balanced", l2_leaf_reg=8,
    random_seed=SEED, verbose=False, allow_writing_files=False
)
v3_failure_full.fit(X_full, y_full, verbose=False)
v3_cost_full = CatBoostRegressor(
    iterations=900, depth=4, learning_rate=0.025,
    loss_function="RMSE", l2_leaf_reg=10,
    random_seed=SEED, verbose=False, allow_writing_files=False
)
v3_cost_full.fit(X_positive_full, train.loc[X_positive_full.index, "repair_cost"].astype(float), verbose=False)
p3_test = v3_failure_full.predict_proba(X_test_full)[:, 1]
r3_test = np.maximum(v3_cost_full.predict(X_test_full), 0)
e3_test = p3_test * r3_test

def make_submission(expected_cost, threshold, filename):
    flags = (expected_cost > threshold).astype(int)
    budgets = np.where(flags == 1, global_q90, 0.0)
    sub = pd.DataFrame({"id": test["id"], "flag": flags, "budget": budgets})
    sub["flag"] = sub["flag"].astype(int)
    sub.to_csv(filename, index=False)
    assert len(sub) == len(test)
    assert sub["id"].is_unique
    assert set(sub["flag"].unique()).issubset({0, 1})
    assert np.isfinite(sub["budget"]).all() and (sub["budget"] >= 0).all()
    print(filename, "| inspections:", int(flags.sum()), "| mean budget among flagged:",
          round(sub.loc[sub.flag == 1, "budget"].mean(), 2) if flags.sum() else 0)
    return sub

sub1 = make_submission(e1_test, threshold1, "submission_v1_baseline.csv")
sub2 = make_submission(e2_test, threshold2, "submission_v2_lightgbm.csv")
sub3 = make_submission(e3_test, threshold3, "submission_v3_catboost.csv")

## 10. Final checks and files

Before submitting, verify:
- exactly 20,000 rows;
- columns are exactly `id, flag, budget`;
- IDs match the test file;
- `flag` contains only 0 or 1;
- budgets are finite and non-negative.

Submit one version at a time to Kaggle. Record the Kaggle score, submission date, Git commit, and the model version.

In [ ]:
for name in ["submission_v1_baseline.csv", "submission_v2_lightgbm.csv", "submission_v3_catboost.csv"]:
    check = pd.read_csv(name)
    print(name, check.shape, list(check.columns),
          "valid flags:", check.flag.isin([0, 1]).all(),
          "valid budgets:", np.isfinite(check.budget).all() and (check.budget >= 0).all())